## Classe pour transformer les fichiers logs en csv 

### Importation des bibliothèques nécessaires

In [75]:
import os
import re
import pandas as pd
import numpy as np
import glob
from pathlib import Path
from datetime import datetime
import logging
from typing import Optional, Dict, List, Tuple
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt


### Configuration logging

In [76]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
 




### Parseur de fichiers journaux Apache au format Combiend Log Format 

In [77]:
class LogParser:
    _LOG_PATTERN = re.compile(
        r'(?P<ip>\d{1,3}(?:\.\d{1,3}){3})'
        r'\s+\S+\s+\S+'
        r'\s+\[(?P<date>[^\]]+)\]'
        r'\s+"(?P<method>\S+)\s+(?P<url>\S+)\s+\S+"'
        r'\s+(?P<status_code>\d{3})'
        r'\s+(?P<response_size>\S+)'
        r'\s+"(?P<referer>[^"]*)"'
        r'\s+"(?P<user_agent>[^"]*)"'
    )

    CHAMPS = [
        "ip", "date", "method", "url",
        "status_code", "response_size", "referer", "user_agent"
    ]

    def __init__(self, log_dir, output_dir=None,pattern="access_log*"):
        self.log_dir = Path(log_dir)
        self.pattern = pattern # Pattern de recherche des fichiers logs

        # Dossier de sortie unique : log_dir/logs_output/
        self.output_dir = Path(output_dir) if output_dir else self.log_dir / "logs_output"
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Fichier CSV consolidé dans le dossier de sortie
        self.output_csv = self.output_dir / "Combined_Access_Logs.csv"

        # Sous-dossier pour les CSV intermédiaires
        self.csv_inter_dir = self.output_dir / "csv_intermediaire"
        self.csv_inter_dir.mkdir(parents=True, exist_ok=True)

        self._frames = {}
        
# Parsing d'une ligne 

    def _parse_line(self, line):
        match = self._LOG_PATTERN.search(line)
        if not match:
            return None
        record = match.groupdict()
        if record["response_size"] == "-":
            record["response_size"] = "0"
        return record
    
# Parsing d'un fichier 
    def parse_file(self, log_file):
        records = []
        with open(log_file, encoding="utf-8", errors="replace") as fh:
            for line in fh:
                rec = self._parse_line(line)
                if rec is not None:
                    records.append(rec)
        return pd.DataFrame(records, columns=self.CHAMPS)
    
# Parsing de tous les fichiers du dossier
    def parse_all(self):
        for log_file in sorted(self.log_dir.glob("*.log")):
            source = log_file.stem
            df = self.parse_file(log_file)
            df["source"] = source
            self._frames[source] = df
            print(f"[OK] {log_file.name:>40s}  →  {len(df):>10,d} requêtes")
        return self._frames
    
 # Exportation des CSV intermédiaires   
    def export_intermediate_csv(self):
        for source, df in self._frames.items():
            out = self.csv_inter_dir / f"{source}.csv"
            df.to_csv(out, index=False)
            print(f"  [CSV] {out}")
            
# Fusion de tous les fichiers 
    def fuse(self):
        if not self._frames:
            raise RuntimeError("Aucun DataFrame. Appelez parse_all() d'abord.")
        combined = pd.concat(self._frames.values(), ignore_index=True)
        combined.to_csv(self.output_csv, index=False)
        print(
            f"\n[FUSION] {len(combined):,d} lignes × "
            f"{len(combined.columns)} colonnes → {self.output_csv}"
        )
        return combined
    

    def summary(self):
        """Affiche un résumé des fichiers produits dans le dossier de sortie."""
        print(f"\n{'='*50}")
        print(f"  RÉSUMÉ DES FICHIERS PRODUITS")
        print(f"  Dossier de sortie : {self.output_dir}")
        print(f"{'='*50}")
        print(f"  1. CSV consolidé     : {self.output_csv}")
        print(f"  2. CSV intermédiaires: {self.csv_inter_dir}/")
        for f in sorted(self.csv_inter_dir.glob("*.csv")):
            print(f"     └── {f.name}")
        print(f"{'='*50}\n")


## Exécution

In [78]:
# ═══ MODIFIEZ CE CHEMIN ═══
LOG_DIR = r"C:\Users\DELL\Desktop\projet\data\Ancien_Site web\httpd\Access_log"
OUTPUT_DIR = r"C:\Users\DELL\Desktop\projet\logs_output\Ancien_Site web\Access_log\csv_intermediaire"
log_parser = LogParser(LOG_DIR, pattern="access_log*")
log_parser.parse_all()


{}